# 🚀 Multi-Agent Fusion Meta-Classifier Training (Colab Drive 142_Extracted)

This notebook trains the final **Logistic Regression / Linear Meta-Classifier** (Fusion Agent) on Google Colab by leveraging prediction scores from all 4 underlying anti-spoofing agents:
1. **Spectral Agent** (XGBoost)
2. **Prosodic Agent** (XGBoost)
3. **Linguistic Agent** (DistilBERT)
4. **SSL Agent** (Pruned WavLM)

### Configured Paths:
* **Linguistic Model**: `/content/drive/MyDrive/142_Extracted/linguistic/model/saved_model`
* **Spectral Model**: `/content/drive/MyDrive/142_Extracted/spectral/model/saved_model/best_spectral_xgb.json`
* **Prosodic Model**: `/content/drive/MyDrive/142_Extracted/prosodic/model/saved_model/best_prosodic_xgb.json`
* **Fusion Dataset**: Saved as `/content/drive/MyDrive/142_Extracted/fusion/dataset/fusion_19k_data.csv`

In [ ]:
# Install all required booster, speech, and helper libraries on Google Colab
!pip install -q xgboost transformers datasets librosa soundfile imbalanced-learn joblib scipy matplotlib seaborn catboost lightgbm

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from google.colab import drive
import random
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Mount Google Drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/142_Extracted')

# 2. Clone Repository for Local Imports
REPO_URL = "https://github.com/saltypal/Multi-Agent-Detection-of-AI-Generated-Speech"
BRANCH = "CoreDevelopment"

!rm -rf Multi-Agent-Detection-of-AI-Generated-Speech
!git clone -b {BRANCH} {REPO_URL}

# 3. Define Project Paths
PROJECT_ROOT = Path("/content/Multi-Agent-Detection-of-AI-Generated-Speech")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"✅ Workspace loaded. Repository root: {PROJECT_ROOT}")

In [ ]:
# 4. Locate and Map your local Drive datasets
DRIVE_ROOT = Path('/content/drive/MyDrive')
BACKUP_DIR = DRIVE_ROOT / '40_per_22' / 'rawbackup'
SPOOF_DIR = DRIVE_ROOT / 'asv_main' / 'test' / 'spoof'
BONAFIDE_DIR = DRIVE_ROOT / 'asv_main' / 'test' / 'bonafide'

# Verify existence
for d in [BACKUP_DIR, SPOOF_DIR, BONAFIDE_DIR]:
    if not d.exists():
        print(f"⚠️ Warning: Directory '{d}' not found on Google Drive.")

# Sample 500 files from each directory as suggested
audio_extensions = ['*.wav', '*.flac', '*.mp3']
random.seed(42)  # For reproducibility

def collect_and_sample(directory, num_samples, class_label=None):
    if not directory.exists():
        return [], {}
    files = []
    for ext in audio_extensions:
        files.extend(list(directory.rglob(ext)))
    
    print(f"[*] Found {len(files):,} audio files in '{directory.name}'.")
    sampled = random.sample(files, min(len(files), num_samples))
    
    lbl_map = {}
    for f in sampled:
        if class_label is not None:
            lbl_map[f.name] = class_label
        else:
            # Dynamically infer label for backup directory based on name keywords
            path_lower = str(f).lower()
            if 'bonafide' in path_lower or 'real' in path_lower:
                lbl_map[f.name] = 0
            elif 'spoof' in path_lower or 'fake' in path_lower:
                lbl_map[f.name] = 1
            else:
                lbl_map[f.name] = 1  # Default fallback
                
    return sampled, lbl_map

sampled_backup, map_backup = collect_and_sample(BACKUP_DIR, 500, class_label=None)
sampled_spoof, map_spoof = collect_and_sample(SPOOF_DIR, 500, class_label=1)
sampled_bonafide, map_bonafide = collect_and_sample(BONAFIDE_DIR, 500, class_label=0)

sampled_files = sampled_backup + sampled_spoof + sampled_bonafide
protocol_map = {**map_backup, **map_spoof, **map_bonafide}

random.shuffle(sampled_files)
print(f"✅ Successfully collected {len(sampled_files)} total files (500 from each of the three directories).")

In [ ]:
# 5. Initialize All 4 Agents on GPU/CPU pointing to custom 142_Extracted paths
from spectral.spectral_model import SpectralAgent
from prosodic.prosodic_model import ProsodicAgent
from linguistic.linguistic_model import LinguisticAgent
from ssl_agent.ssl_model import SSLAgent

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing Agents on device: {device}...")

# Point agents to the specific 142_Extracted model directories
spec_agent = SpectralAgent(BASE_DIR / "spectral" / "model" / "saved_model")
pros_agent = ProsodicAgent(BASE_DIR / "prosodic" / "model" / "saved_model")
ling_agent = LinguisticAgent(BASE_DIR / "linguistic" / "model" / "saved_model", device=device)
ssl_agent  = SSLAgent("JYP2024/Wedefense_ASV2025_WavLM_Base_Pruning", device=device)

print("✅ All agents successfully loaded from 142_Extracted on Google Drive!")

In [ ]:
# 6. Run Multi-Agent Parallel Inference Pipeline
results = []

for path in tqdm(sampled_files, desc="Running Agent Inference"):
    try:
        # Predict probabilities (0 = Bonafide, 1 = Spoof)
        p_spec = spec_agent.predict(path)
        p_pros = pros_agent.predict(path)
        
        try:
            p_ling = ling_agent.predict(path)
        except Exception as e:
            p_ling = 0.0 # fallback
            
        try:
            p_ssl = ssl_agent.predict(path)
        except Exception as e:
            p_ssl = 0.5 # fallback
        
        results.append({
            'filename': path.name,
            'P_spec': p_spec,
            'P_pros': p_pros,
            'P_ling': p_ling,
            'P_ssl':  p_ssl,
            'label': protocol_map[path.name]
        })
    except Exception as e:
        print(f"⚠️ Skipped {path.name} due to feature extraction error: {e}")

fusion_df = pd.DataFrame(results)

# Save predictions to fusion_19k_data.csv in fusion/dataset as requested
FUSION_DIR = BASE_DIR / 'fusion' / 'dataset'
FUSION_DIR.mkdir(parents=True, exist_ok=True)
fusion_df.to_csv(FUSION_DIR / "fusion_19k_data.csv", index=False)

print(f"✅ Inference Pipeline Completed. Saved scores to Google Drive at: {FUSION_DIR / 'fusion_19k_data.csv'}")

In [ ]:
# 7. Train the Meta-Classifier & Trust Weighting
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import joblib
from scipy.optimize import brentq
from scipy.interpolate import interp1d

X = fusion_df[['P_spec', 'P_pros', 'P_ling', 'P_ssl']].values
y = fusion_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit linear log-odds meta-classifier
meta_model = LogisticRegression(class_weight='balanced')
meta_model.fit(X_train, y_train)

print("\n" + "="*50 + "\n🏆 MULTI-AGENT TRUST COEFFICIENTS (WEIGHTS)\n" + "="*50)
agents = ['Spectral Agent', 'Prosodic Agent', 'Linguistic Agent', 'SSL WavLM Agent']
weights = meta_model.coef_[0]

for name, coef in zip(agents, weights):
    print(f"  - {name:<20} Trust Coefficient: {coef:+.4f}")

# Evaluate Ensemble on Holdout dev split
y_prob = meta_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# Calculate EER
fpr, tpr, _ = roc_curve(y_test, y_prob)
fnr = 1 - tpr
eer = brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
auc = roc_auc_score(y_test, y_prob)

print("\n" + "="*50 + "\n📊 FUSION ENSEMBLE PERFORMANCE ON HOLDOUT DEV SET\n" + "="*50)
print(f"  - Final Ensemble EER : {eer*100:.2f}%")
print(f"  - Final Ensemble AUC : {auc:.4f}")
print(f"\nEnsemble Classification Report:\n", classification_report(y_test, y_pred))

# Save Meta-Model to Google Drive
FUSION_MODEL_DIR = BASE_DIR / 'fusion' / 'model' / 'saved_model'
FUSION_MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(meta_model, FUSION_MODEL_DIR / "fusion_meta_model.pkl")

print(f"🎉 Meta-Classifier successfully trained and saved to Google Drive at: {FUSION_MODEL_DIR / 'fusion_meta_model.pkl'}!")